# Séance 2 – Algorithmie : rappel des bases 

**Enseignant :** Jean Delpech

**Cours :** Algorithmie et développement dans l'ingénierie des données

**Classe :** M1 Data (IA/IE)

**Année scolaire :** 2025/2026

**Dernière mise à jour :** avril 2026

---

## Objectifs de la séance

- Savoir lire et mener une analyse de complexité en notation Big O
- Comprendre pourquoi la complexité a des conséquences **concrètes et mesurables** en data science
- Identifier les différences de performance entre les structures de données de base : `list`, `dict`, `set`, `numpy.ndarray`

---

## Plan

| # | Section |
|---|---|
| 1 | Complexité algorithmique – théorie |
| 2 | Arrays, mémoire et vectorisation |
| 3 | Tables de hachage |
| 4 | Exercices pratiques guidés | 

# PARTIE 1 – Complexité algorithmique
## 1.1 Pourquoi ça compte en Data Science ?

La complexité algorithmique décrit **comment le temps d'exécution (ou la mémoire utilisée) évolue avec la taille des données**.

En data science, les volumes sont souvent grands. Une différence de complexité qui semble théorique peut représenter la différence entre un traitement en **3 secondes** et un traitement en **plusieurs heures**.

### Notation Big O

La notation **O(f(n))** décrit le comportement **dans le pire cas** et **asymptotiquement** (pour de grandes valeurs de n).

On ignore les constantes : O(3n) = O(n)

On ne retient que le terme dominant : O(n² + n) = O(n²).

| Complexité | Nom | Exemple |
|---|---|---|
| O(1) | Constante | Accès à un élément d'un dict par clé |
| O(log n) | Logarithmique | Recherche dans un arbre binaire équilibré |
| O(n) | Linéaire | Parcourir une liste |
| O(n log n) | Quasi-linéaire | Tri (Mergesort, Timsort) |
| O(n²) | Quadratique | Double boucle imbriquée |
| O(2ⁿ) | Exponentielle | Exploration exhaustive de combinaisons |

In [ ]:
# Visualisation des classes de complexité
import numpy as np
import matplotlib.pyplot as plt

n = np.linspace(1, 50, 500)

plt.figure(figsize=(10, 6))
plt.plot(n, np.ones_like(n),       label='O(1)',       linewidth=2)
plt.plot(n, np.log2(n),            label='O(log n)',   linewidth=2)
plt.plot(n, n,                     label='O(n)',       linewidth=2)
plt.plot(n, n * np.log2(n),        label='O(n log n)', linewidth=2)
plt.plot(n, n**2,                  label='O(n²)',      linewidth=2)
plt.plot(n[n<=20], 2**n[n<=20],    label='O(2ⁿ)',      linewidth=2, linestyle='--')

plt.ylim(0, 2500)
plt.xlabel('Taille des données n')
plt.ylabel('Opérations')
plt.title('Classes de complexité – comparaison visuelle')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Impact concret sur des volumes data réels

Prenons un exemple : **une jointure sans index** est O(n²). Avec un index (tri + fusion), elle passe à O(n log n).

**Jointure sans index – O(n²)**

Une jointure consiste à relier deux tables sur une colonne commune (ex. `ON commandes.client_id = clients.id`). Sans index, la stratégie naïve est la **nested loop join** : pour chaque ligne de la première table, on parcourt intégralement la deuxième table pour trouver les correspondances. Si les deux tables ont n lignes, on effectue n × n comparaisons – d'où O(n²).

**Jointure avec index – O(n log n)**

Un index sur la colonne de jointure est une structure de données triée (typiquement un B-tree, structure qu’on verra plus tard) qui permet de localiser rapidement les lignes correspondantes sans parcourir toute la table. La stratégie utilisée est alors le **sort-merge join** : on trie les deux tables sur la colonne de jointure (O(n log n) chacune), puis on les fusionne en un seul parcours linéaire (O(n)) en avançant deux pointeurs en parallèle – exactement comme la phase de fusion du tri fusion. Le coût total est dominé par le tri : O(n log n).

> Si l'index existe déjà (ce qui est son intérêt principal), le coût du tri est évité et la jointure se réduit à la fusion seule : O(n).

Calculons ce que ça représente en nombre d'opérations :

In [ ]:
import math

tailles = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]

print(f"{'Taille n':>12} | {'O(n log n)':>15} | {'O(n²)':>18} | {'Ratio':>10}")
print("-" * 65)
for n in tailles:
    nlogn = n * math.log2(n)
    n2 = n ** 2
    ratio = n2 / nlogn
    print(f"{n:>12,} | {nlogn:>15,.0f} | {n2:>18,} | {ratio:>10,.0f}x")

### Complexité en espace

La complexité **spatiale** décrit la mémoire consommée. En data science, c'est souvent le vrai goulot d'étranglement :
- Un DataFrame de 1M lignes × 100 colonnes float64 = ~800 Mo
- Une matrice de similarité entre 100K documents = 100K² × 8 octets = **80 Go** !

**Règle pratique :** avant d'écrire un algorithme, estimer si le résultat intermédiaire tient en RAM.

## Impact des structures de contrôle sur la complexité

| Structure | Effet sur la complexité | Exemple | Complexité résultante |
|---|---|---|---|
| Opération simple | Aucun – facteur constant | `x = a + b` | O(1) |
| Séquence d'instructions | Addition – on garde le terme dominant | O(n) + O(n²) | O(n²) |
| `if / else` | On retient le pire cas des branches | `if ... else ...` | max(O(branche 1), O(branche 2)) |
| Boucle `for` / `while` simple | Multiplication par le nombre d'itérations | `for i in range(n)` | O(n) × O(corps) |
| Boucles imbriquées | Multiplication des facteurs | `for i` dans `for j` | O(n) × O(m) = O(n·m) |
| Récursion simple (1 appel) | Profondeur de récursion × coût par appel | `f(n) = f(n-1) + O(1)` | O(n) |
| Récursion arborescente (2 appels) | Exponentielle si non mémoïsée | `f(n) = f(n-1) + f(n-2)` | O(2ⁿ) |
| Division par 2 à chaque étape | Logarithmique | Recherche binaire | O(log n) |
| Boucle + division par 2 | Linéarithmique | Tri fusion | O(n log n) |
| Slicing Python `l[a:b]` | Copie proportionnelle à la taille du slice | `l[:n]` | O(n) |
| `in` sur une liste | Parcours linéaire | `x in my_list` | O(n) |
| `in` sur un set / dict | Hachage – temps constant | `x in my_set` | O(1) |

> **Règle d'or** : pour estimer la complexité d'un algorithme, repère d'abord les boucles et les appels récursifs – ce sont eux qui font exploser le coût. Les opérations atomiques (affectation, comparaison, accès à un index) sont toujours O(1) et s'absorbent dans les constantes.

Efforcez-vous de faire les exercices proposés sans demander à l’IA. Utilisez votre cerveau. Vous êtes en phase d’apprentissage, ce qui a pour objectif de développer votre intellect, pas de l’atrophier, ce que ne manquera pas de faire le recours à un outil qui vous évite de réfléchir.

Maîtrisez votre usage de l’IA. Vous pouvez lui demander des indices, des explications, de relire votre code, d’améliorer la forme, de vous expliquer pourquoi une solution est meilleure qu’une autre, et même de vous proposer d’autres exercices…

![Don’t do drugs / don’t do vibe-coding](./Images/Dont.png)

### Exercice 1 – Boucle simple

Estimez la complexité de la fonction suivante, puis mesurez son temps d'exécution pour `n ∈ [100, 1000, 10000, 100000, 1000000]`. (pour mieux visualiser vous pouvez tracer la courbe temps d’éxécution en fonction de $n$)

Que remarquez-vous sur la relation entre n et le temps d'exécution ?

In [ ]:
import time
import matplotlib.pyplot as plt

def somme_lineaire(n):
    total = 0
    for i in range(n):
        total += i
    return total

# Votre code ici

tailles = 
temps =

plt.plot(tailles, temps, marker='o')
plt.xlabel("n")
plt.ylabel("Temps (s)")
plt.title("Complexité mesurée – Exercice 1")
plt.grid(True)
plt.show()

### Exercice 2 – Boucles imbriquées

Estimez la complexité de la fonction suivante, puis mesurez-la pour `n ∈ [50, 100, 200, 500, 1000]`.

Comparez graphiquement avec la courbe $n^2$ théorique.

In [ ]:
import numpy as np

def paires(n):
    resultat = []
    for i in range(n):
        for j in range(n):
            resultat.append((i, j))
    return resultat

# Votre code ici

tailles = 

### Exercice 3 – Recherche dans une liste vs un set

La fonction `in` n'a pas le même coût selon la structure utilisée.

- Mesurez le temps de recherche d'un élément **absent** dans une liste, puis dans un set, pour `n ∈ [1000, 10000, 100000, 1000000]`.
- Que constatez vous ?

In [ ]:
def recherche_liste(n):
    lst = list(range(n))
    return -1 in lst  # élément absent : pire cas

def recherche_set(n):
    s = set(range(n))
    return -1 in s

# Votre code ici

tailles = 

### Exercice 4 – Recherche binaire

La fonction suivante implémente une recherche binaire dans une liste triée.

1. Estimez sa complexité
2. Mesurez-la pour `n ∈ [100, 1000, 10000, 100000, 1000000, 10000000]`
3. Superposez la courbe log(n) théorique – qu'observez-vous sur l'échelle des temps ?

In [ ]:
import math

def recherche_binaire(lst, cible):
    gauche, droite = 0, len(lst) - 1
    while gauche <= droite:
        milieu = (gauche + droite) // 2
        if lst[milieu] == cible:
            return milieu
        elif lst[milieu] < cible:
            gauche = milieu + 1
        else:
            droite = milieu - 1
    return -1

# Votre code ici

tailles = 

# PARTIE 2 – Arrays et mémoire

## 2.1 Pourquoi `numpy.ndarray` est plus rapide qu'une `list` Python

### Modèle mémoire d'une liste Python

Une `list` Python est une liste de **pointeurs** vers des objets Python. Chaque élément est un objet Python complet (avec type, référence, valeur).

```
list = [ ptr → PyObject, ptr → PyObject, ptr → PyObject, ... ]
```

**Conséquences :**
- Les éléments ne sont **pas contigus** en mémoire → les caches CPU sont peu efficaces
- Chaque opération nécessite de **déréférencer un pointeur** → overhead constant (cf. ci-dessous)

### Modèle mémoire d'un `numpy.ndarray`

Un tableau numpy stocke les données de façon **contiguë** en mémoire, avec un **type uniforme** (dtype).

```
ndarray = [ float64 | float64 | float64 | float64 | ... ] (bloc mémoire contigu)
```

**Conséquences :**
- Les caches CPU fonctionnent efficacement (préchargement)
- Les opérations vectorisées utilisent les **instructions SIMD** du processeur
- Pas d'overhead d'interprétation Python boucle par boucle

On a fait appel a plusieurs concepts ou techniques qui méritent d’être expliquées :

#### « Overhead d'interprétation », qu’es aquò ? 

Il ne faut pas oublier que Python est un langage interprété, ce qui pose quelque contraintes/limitation lors de l’exécution.
En Python pur, chaque itération d'une boucle `for` a un coût fixe incompressible : l'interpréteur doit à chaque tour vérifier le type de chaque variable, dispatcher vers la bonne opération, gérer le compteur de références pour le garbage collector, etc. Ce surcoût par itération s'appelle l'**overhead d'interprétation**.

Sur une boucle de 1 000 000 d'éléments, cet overhead se répète 1 000 000 de fois – même si l'opération utile (une addition) prend une nanoseconde, l'overhead peut représenter 10 à 100 fois ce coût.

NumPy élimine cet overhead en déplaçant la boucle dans du code C compilé : l'interpréteur Python est appelé **une seule fois** pour déclencher l'opération, puis C itère sur le tableau sans aucune intervention de Python.

#### Caches CPU et préchargement

Un tableau NumPy est un **bloc mémoire contigu** : tous les éléments sont rangés côte à côte en RAM. Le processeur peut donc anticiper les accès et charger les prochains éléments dans son cache avant même qu'ils soient demandés (mécanisme de *prefetch*).

Une liste Python, à l'inverse, est un tableau de **pointeurs** vers des objets éparpillés en mémoire. Chaque accès peut provoquer un *cache miss* – le CPU doit aller chercher la donnée en RAM, ce qui est 10 à 100 fois plus lent qu'un accès en cache.

#### Instructions SIMD

Les processeurs modernes disposent d'instructions capables d'effectuer la **même opération sur plusieurs valeurs simultanément** (*[single instruction multiple data – SIMD](https://en.wikipedia.org/wiki/Single_instruction,_multiple_data)* : SSE, AVX sur x86) à l’aide de registres dédiés. Par exemple, une instruction AVX-256 peut additionner 8 flottants 32 bits en un seul cycle d'horloge au lieu de 8. Aujourd’hui on en est à l’AVX-512.

NumPy, compilé avec optimisations, exploite ces instructions automatiquement sur ses tableaux contigus. Une boucle Python ne peut pas en bénéficier, car le CPU ne peut pas vectoriser des opérations dont les données sont éparpillées en mémoire.

### Exemple et exercice

#### Temps d’exécution

Comparez le temps d’exécution pour sommer 1 000 000 d’éléments placés dans une liste vs. une ndarray :

In [ ]:
import numpy as np
import timeit

N = 1_000_000

# VOTRE CODE




# Somme avec une liste Python (boucle)
t_list = 

# Somme avec numpy (vectorisé)
t_numpy = 

print(f"Taille : {N:,} éléments")
print(f"sum() sur liste Python : {t_list*1000:.2f} ms")
print(f"np.sum() sur ndarray   : {t_numpy*1000:.2f} ms")
print(f"Accélération           : x{t_list/t_numpy:.0f}")

#### Mémoire

Comparez la place (approximative) prise en mémoire par une liste de n éléments placés dans une liste vs. une ndarray.

* utilisez la méthode `sys.getsizeof()` pour estimer la taille de la liste ([n’oubliez pas qu’une liste, même vide, occupe de la mémoire](https://stackoverflow.com/questions/7247298/size-of-a-python-list-in-memory)). [Attention ne confondez pas `len()` et `.getsizeof()`](https://www.delftstack.com/fr/howto/python/difference-between-len-and-getsizeof-in-python/)
* les objets `ndarray` disposent de l’attribut `.nbytes` ([RTFM](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.nbytes.html))

In [ ]:
# Comparaison mémoire
import sys

n = 10_000

# VOTRE CODE 

lst = 
arr = 

# Taille mémoire approximative
size_list = 
size_arr  = 

print(f"list Python ({n} entiers) : {size_list / 1024:.1f} Ko")
print(f"numpy array ({n} int64)   : {size_arr  / 1024:.1f} Ko")
print(f"Ratio mémoire             : x{size_list/size_arr:.1f}")

### Matrices creuses avec `scipy.sparse`

En NLP ou en recommandation, les matrices sont souvent **creuses** (*sparse*) : 99% des valeurs sont zéro.

#### Exercice : Matrices creuses et empreinte mémoire

On considère une matrice TF-IDF de 10 000 documents × 50 000 mots avec une densité de 0.1% (la grande majorité des mots n'apparaissent pas dans chaque document).

**Étape 1 – Estimer la taille d'une matrice dense**

Sans créer la matrice (elle ferait ~4 Go), calculez `size_dense_GB` à partir des dimensions et du fait qu'un flottant 64 bits occupe 8 octets.

**Étape 2 – Créer la matrice creuse et mesurer sa taille réelle**

Une matrice au format CSR (Compressed Sparse Row) stocke trois tableaux :
- `data` : les valeurs non nulles
- `indices` : les indices de colonne de chaque valeur non nulle
- `indptr` : les pointeurs de début de chaque ligne

Calculez `size_sparse_MB` en sommant les tailles en octets de ces trois tableaux (attribut `.nbytes`).

**Étape 3 – Calculer le facteur de compression**

Déduisez-en le rapport entre les deux tailles.

Exemple : matrice document × mots pour 10 000 documents et un vocabulaire de 50 000 mots.  
→ Taille dense : 10 000 × 50 000 × 8 octets = **4 Go**  
→ Taille creuse (si 0.1% de valeurs non nulles) : ~**4 Mo**

In [ ]:
from scipy.sparse import random as sparse_random
import numpy as np

n_docs, n_words = 10_000, 50_000
density = 0.001

sparse_mat = sparse_random(n_docs, n_words, density=density, format='csr')

# Étape 1 – taille dense estimée (en Go)
# un float64 occupe 8 octets, 1 Go = 1e9 octets
size_dense_GB = 

# Étape 2 – taille creuse réelle (en Mo)
# accédez aux attributs .data, .indices, .indptr de sparse_mat
# rappel : 1 Mo = 1e6 octets
size_sparse_MB = 

print(f"Dimensions : {n_docs:,} × {n_words:,}")
print(f"Densité    : {density*100:.1f}%")
print(f"Taille dense (estimée)  : {size_dense_GB:.1f} Go")
print(f"Taille creuse (réelle)  : {size_sparse_MB:.1f} Mo")
print(f"Facteur de compression  : x{size_dense_GB*1000/size_sparse_MB:.0f}")

# PARTIE 3 – Tables de hachage

## 3.1 Rappels – Structures de données fondamentales

Avant d'analyser la complexité des opérations sur les structures Python, il est utile de rappeler un peu plus précisément deux mécanismes sous-jacents qui les implémentent.

### Tableau contigu (array)

Un tableau contigu stocke ses éléments **côte à côte en mémoire**, à des adresses consécutives. 

L'accès à un élément par son index est immédiat : l'adresse de l'élément i se calcule directement comme `adresse_base + i × taille_élément`. C'est pourquoi `mon_tableau[i]` est toujours O(1).

![Schéma tableau contigu](./Images/Index.png)

En contrepartie, **insérer ou supprimer un élément** au milieu oblige à décaler tous les éléments suivants : O(n).

### Liste chaînée (linked list)

Une liste chaînée ne stocke pas ses éléments de manière contiguë. Chaque élément est un **nœud** qui contient la valeur et un pointeur vers le nœud suivant. Les nœuds peuvent être éparpillés n'importe où en mémoire.

![Schéma liste chaînée](./Images/ListeChainee.png)

**Insérer ou supprimer** un élément est O(1) si on est déjà positionné au bon endroit (on redirige juste un pointeur). En revanche, **accéder au i-ème élément** oblige à parcourir la chaîne depuis le début : O(n).

> Attention à la confusion par rapport à ce qu’on a dit plus tôt sur les `list` Python dont les valeurs ne sont pas stockées en mémoire de manière contigue : les `list` ne sont **pas** pour autant des listes chaînées : ce sont des tableaux dynamiques de pointeurs. Les listes chaînées apparaissent en Python dans d'autres contextes (ex. `collections.deque`) et nous les reverrons plus en détail ultérieurement.
> Pour ce qui est des `list` voyez une présentation plus détaillées dans la sous-section `Tableau dynamique vs liste chaînée` ci-dessous.


### Tableau dynamique vs liste chaînée

Ces deux structures offrent toutes deux une collection ordonnée d'éléments, mais avec des compromis très différents.

#### Tableau dynamique

Un tableau dynamique est un tableau contigu dont la taille peut grandir. Quand le tableau est plein et qu'on ajoute un élément, Python alloue un nouveau bloc mémoire plus grand (en général le double), copie tous les éléments, puis libère l'ancien bloc. Cette opération de redimensionnement est O(n), mais elle est suffisamment rare pour que le coût moyen d'un `append` reste **O(1) amorti**.

Une `list` est un tableau dynamique.

#### Pourquoi ne peut-on pas vectoriser une `list` Python ?

La contiguïté d'une `list` Python est trompeuse : ce qui est stocké côte à côte en mémoire, ce ne sont pas **les valeurs** elles-mêmes, mais des **pointeurs** vers des objets Python éparpillés dans le tas (heap).

![Schéma list](./Images/List.png)

Et il y a autre chose qui complexifie encore la situation : tout en Python est objet, même un type aussi simple qu’un entier. Ainsi chaque objet `int` Python n'est pas un simple entier, c'est une structure qui embarque un type, un compteur de références, et la valeur. Cela a deux conséquences :

- **Pas de vectorisation SIMD** : les instructions SIMD exigent des valeurs numériques brutes, alignées et de taille fixe en mémoire. Les objets Python ne satisfont pas ces contraintes.
- **Cache miss systématique** : accéder aux valeurs oblige le CPU à suivre les pointeurs vers des adresses mémoire imprévisibles et on retombe exactement sur le problème que pose l’emploie d’une liste chaînée.

Un `ndarray` NumPy stocke au contraire les **valeurs brutes** (ex. 8 octets par float64) directement côte à côte, sans redirection. C'est cette différence « valeurs brutes vs pointeurs » qui rend NumPy vectorisable et non la simple contiguïté.

> C'est aussi pourquoi un `ndarray` est **homogène** (contient qu’un seul type) : stocker des valeurs brutes sans métadonnées de type impose que tous les éléments aient exactement la même représentation (et taille) en mémoire.

#### Liste chaînée

Chaque ajout alloue simplement un nouveau nœud et redirige un pointeur. Il n’y a donc pas de redimensionnement à la volée, ni copie. L'insertion est toujours O(1) exactement, sans notion d'amorti.

#### Complexité des opérations

| Opération | Tableau dynamique | Liste chaînée |
|---|---|---|
| Accès par index `[i]` | O(1) | O(n) |
| Insertion/suppression en fin | O(1) amorti | O(1) |
| Insertion/suppression au milieu | O(n) | O(1) * |
| Empreinte mémoire | Faible (données seules) | Plus élevée (données + pointeurs) |
| Localité mémoire (cache CPU) | Excellente | Mauvaise |

*\* à condition d'être déjà positionné au bon nœud, sinon O(n) pour y accéder*

En pratique, la mauvaise localité mémoire des listes chaînées les pénalise fortement sur le matériel moderne (cache, parallélisme des opérations…). Les tableaux dynamiques sont presque toujours plus rapides en Python, même pour des opérations théoriquement avantageuses pour les listes chaînées.

> Attention de pas généraliser abusivement. On le verra plus tard, mais par exemple `collections.deque` en Python est implémenté comme une liste doublement chaînée de blocs, ce qui lui permet d'offrir O(1) en insertion/suppression aux **deux extrémités**, c'est le cas d'usage principal.

### Table de hachage (hash table)

Une table de hachage associe des **clés** à des **valeurs** en s'appuyant sur une fonction de hachage `hash(clé)` qui transforme la clé en un entier, lequel sert d'index dans un tableau interne.

#### Les fonctions de hachage

Une fonction de hachage prend une donnée de taille arbitraire (une chaîne de caractères, un entier, un tuple...) et produit un entier de taille fixe, appelé **hash** ou **empreinte**. En Python, `hash("alice")` retourne par exemple `-4670613186496113837`, un entier qui représente la `string` "alice" de manière compacte. Dans ce cas par exemple, le calcul repose sur les octets bruts de la donnée : pour une chaîne, on combine les codes ASCII de chaque caractère via des opérations arithmétiques et des décalages de bits, de façon à ce que des entrées similaires produisent des hashs très différents (`"alice"` et `"Alice"` donnent des hashs sans aucun rapport :`hash(Alice) = 2428297203169026435`).

Une bonne fonction de hachage doit respecter plusieurs contraintes :

- **Déterministe** : la même entrée produit toujours le même hash, c’est une condition indispensable pour retrouver une clé.
- **Uniforme** : les hashs doivent se répartir de façon homogène sur l'espace disponible, pour minimiser les collisions.
- **Rapide à calculer** : le calcul du hash doit être O(1) ou quasi O(1), sinon l'avantage de la table de hachage disparaît. En pratique le coût est proportionnel à la taille de la donnée hachée (O(k) pour une chaîne de k caractères), ce qui reste négligeable pour des clés de taille raisonnable.
- **Effet avalanche** : un changement minime dans l'entrée doit produire un hash radicalement différent. C'est ce qui garantit une bonne distribution et limite les collisions.

En Python **seuls les objets immuables sont hashables**. Si un objet pouvait changer après avoir été inséré dans un `dict` ou un `set`, son hash changerait et il deviendrait introuvable la table de hachage serait corrompue. C'est pourquoi `hash([1, 2, 3])` lève une `TypeError` : une liste est mutable, donc non hashable.

## 3.2 Principe

Une **table de hachage** stocke des paires (clé, valeur) et permet un accès en **O(1) amorti**.

L’idée centrale est d’éviter une recherche séquentielle dans toutes les données.
Au lieu de parcourir le tableau case par case, on calcule directement l’emplacement où la valeur doit être stockée grâce à une fonction de hachage.

### Mécanisme

La fonction de hachage transforme une clé (par exemple une chaîne de caractères ou un entier) en un nombre entier.
Ce nombre est ensuite converti en un indice valide du tableau interne en applicant une fonction de compression.

```
clé  →  hash(clé)  → fonction de compression → indice dans un tableau  →  valeur
```

On calcule un hash à partir de la clé.
On doit ensuite ramener ce hash (qui est un nombre très grand) à une case du tableau (son indice, qui est un nombre généralement plus petit avec un nombre de possibilité plus réduite). 
Pour cela, comme le tableau a une dimension finie, on calcule en général l’indice de la case désiré avec un modulo (notre fonction de compression) :

```
indice_h = hash(cle) mod taille_tableau
```

On peut ainsi accèder directement à `tableau[h]`.

Par exemple, si la clé est `"alice"` :
```
h = hash("alice") % 100
```
et que le résultat vaut 42, alors la valeur associée sera stockée dans `tableau[42]`.



In [ ]:
# Démonstration : hash() en Python
exemples = ["alice", "bob", "alice", 42, (1, 2), 3.14]

print("Valeurs de hash en Python :")
for e in exemples:
    print(f"  hash({str(e)}) = {hash(e)}")

print("\nNote : les listes ne sont pas hashables (mutables)")
try:
    hash([1, 2, 3])
except TypeError as e:
    print(f"  hash([1,2,3]) → TypeError: {e}")

**Exercice** : supposons que vous voudriez entrer les valeurs de l’exemple précédent dans un tableau de taille 10. Calculez les indices avec une fonction de compression (un modulo peut suffire) et entrez les valeurs dans ce tableau :

In [ ]:
# Votre code ici



#### Efficacité

Cette approche est avantageuse pour retrouver une valeur dans un ensemble de données. Si on compare avec un tableau classique non trié, rechercher une valeur peut nécessiter de parcourir toutes les cases du tableau :

* meilleur cas : immédiat
* pire cas : O(n)

Avec une table de hachage, la position est calculée directement à partir de la clé. La complexité est donc constante O(1).
On évite donc la recherche linéaire : l’accès est généralement quasi instantané, même lorsque la structure contient beaucoup d’éléments.

C’est ce qui rend les dictionnaires Python (`dict`) ou les tables associatives très performantes.

#### Retour sur l'exercice 3 – Pourquoi le set est-il O(1) ?

Dans l'exercice 3 (au tout début de ce notebook), vous avez observé que la recherche dans un `set` reste quasiment instantanée quelle que soit la taille de la collection, là où la liste croît linéairement.

Cette différence s'explique par le **hachage** : un `set` Python est implémenté comme une **table de hachage**. Lorsqu'on teste `x in my_set`, Python calcule `hash(x)` et accède directement à l'emplacement mémoire correspondant sans parcourir les autres éléments. Le nombre d'éléments dans le set n'a aucune influence sur ce calcul.

C'est exactement la même mécanique qui rend les lookups dans un `dict` O(1) : la clé est hachée pour localiser la valeur associée en temps constant.

> La contrepartie : seuls les objets **hashables** (immuables) peuvent être placés dans un `set` ou utilisés comme clé de `dict`. Une liste Python, étant mutable, n'est pas hashable – `{[1, 2]: "valeur"}` lèvera une `TypeError`. > La contrepartie : seuls les objets **hashables** (immuables) peuvent être placés dans un `set` ou utilisés comme clé de `dict`. De même `{[1, 2, 3]}` lèvera une `TypeError`. C'est pourquoi on utilise des tuples lorsqu'on a besoin de stocker une séquence dans un set : `{(1, 2, 3)}` est valide.

#### Complexité

Dire qu’une opération est en O(1) signifie que son coût reste constant : le temps d’accès ne dépend pas du nombre d’éléments stockés.

En pratique, les opérations classiques :

* insertion
* recherche
* suppression

sont généralement très rapides.

On parle cependant de O(1) amorti plutôt que simplement O(1) car certaines opérations exceptionnelles peuvent être plus coûteuses.

Par exemple, lorsque le tableau devient trop rempli (on parle « d’augmentation de la charge »), la table de hachage doit :

1. créer un tableau plus grand
2. recalculer la position des éléments 
3. recopier toutes les données

Cette opération de redimensionnement coûte temporairement O(n), mais elle reste rare.
Si l’on considère une longue suite d’opérations, le coût moyen par opération reste constant : on dit donc que la complexité est amortie.

Cette approche possède néanmoins un point faible : les collisions.

### Collisions

Nous venons de voir que la fonction de hachage produit souvent un très grand entier. Cet entier est ensuite ramené à une case du tableau avec une fonction de « compression » (par exemple avec un modulo).
Il y a moins de cases possibles du tableau que le nombre de hash (souvent un entier de 64 bits) qui peut être généré. Le modulo va permettre de réduire cet ensemble de possibilités à un nombre de cases plus réduit. Mais cela va nous amener dans une situation problématique :

>Deux clés différentes peuvent être associées à la même case du tableau après application de la fonction de compression aux deux hashs correspondants : c'est une **collision**.

Quelle que soit la qualité de la fonction de hachage, les collisions sont inévitables. En effet, le nombre de clés possibles est généralement immense (voire infini), alors que le tableau interne possède un nombre fini de cases. Plusieurs clés distinctes finiront donc nécessairement par être associées à la même case du tableau. Deux grandes stratégies permettent de gérer ces collisions.

#### Exemple

Supposons une table de hachage de taille 10 (c.-à-d. 10 cases) et une fonction de hachage très simple :

$$
h(x) = x \mod 10
$$

* la clé 25 donne $25 \mod 10 = 5$
* la clé 35 donne aussi $35 \mod 10 = 5$

Les deux clés veulent donc être stockées dans la case d’indice 5 : il y a collision. 

Pour fixer les idées, une table de hachage fonctionne un peu comme un parking avec un nombre limité de places : même si tout se passe bien la plupart du temps, il va forcément y avoir un moment où deux voitures voudront la même place.

On met couramment en œuvre deux solutions possibles à ce problème :
- le **chaînage** : chaque case contient une liste chaînée de paires (clé, valeur)
- l’**adressage ouvert** : on cherche la prochaine case libre (*linear probing*)

En quoi consistent-elles ?

#### Chaînage

L’approche de cette mémthode est qu’on ne va pas stocker dans chaque case du tableau une unique valeur à laquelle on va accéder directement, mais une structure capable de stocker plusieurs éléments qu’on va pouvoir insérer à la volée. Généralement on va utiliser une **liste chaînée** de toutes les paires (clé, valeur) dont le hash pointe vers cette case.

Voici l’exemple d’une table avec une collision sur la case 2 :
| Indice | Valeur |
|--------|--------|
| [0] | → None|
| [1] | → None|
| [2] | → ("alice", 42) → ("charlie", 7) → None|
| [3] | → ("bob", 15) → None|
| [4] | → None|

Ici `'alice'` et `'charlie'` ont produit le même indice (`2`), ils sont donc stockée dans la même liste chaînée.

Impact sur les opérations classique :
* Insertion : on calcule le hash qui donne l’indice, on ajoute l’élément dans la liste chaînée correspondant à cet indice, généralement en tête de liste. La complexité de l’insertion est *en moyenne* en O(1), car l’ajout en tête de liste est constant et les listes restent généralement courtes lorsque la fonction de hachage répartit bien les clés.
* Recherche : on calcule le hash, puis on parcourt la liste chaînée en comparant les clés. La complexité va de O(1) si la liste est courte à O(n) dans le pire cas (l’élément n’est pas dans la liste donc il faut parcourir toute la liste pour se rendre compte qu’il est absent) si toutes les clés collisionnent sur la même case.
* Suppresion : idem recherche.

### Facteur de charge

On peut introduire ici la notion de *facteur de charge* $\alpha$, qui va indiqué « à quel point la table est remplie ». Sa formule est très simple :

$$
\alpha = \frac{n}{m}
$$
Avec :
- $n$ : nombre d’éléments stockés
- $m$ : nombre de cases dans la table (taille de la table)

Plus le facteur de charge augmente, plus les collisions deviennent fréquentes et plus les listes chaînées s’allongent.

$\alpha$ est donc lié à :
- la longueur moyenne des listes chaînées
- la probabilité de collision
- le moment du redimensionnement (augmenter dynamiquement la taille de la table)

Quand le tableau est trop plein (facteur de charge > 0.7 environ), Python **redimensionne** automatiquement le dict (×2) → coût O(n) amorti sur n insertions, soit O(1) par insertion.

Si la taille du tableau est bien adaptée et si la fonction de hachage répartit uniformément les clés, alors les listes chaînées restent courtes en moyenne (et si non cela signifie qu’il faut déclencher un redimensionnement, ou alors que la fonction de hachage n’est pas adatpée). **Maintenir des listes chaînées courtes est ce qui garantie le O(1) moyen**.

## 3.2 Implémentation from scratch d'une table de hachage

On implémente une table de hachage simple avec **chaînage** pour gérer les collisions.

Lisez le code, et essayez de le comprendre.

Ensuite :

1. écrivez un test pour vérifier ce qu’il se passe quand on entre des valeurs (par exemple quelques prénoms) dans une table de taille modeste (n=8), récupérez des valeurs, affichez le facteur de charge, les collisions…
2. entrez plus de prénoms dans la table par rapport à sa taille (pour augmenter le facteur de charge). Avec MatplotLib visualisez la répartition dans la table : y a-t-il des emplacements (buckets) où il y a plus de valeurs que d’en d’autres ?

In [ ]:
class HashTable:
    """Table de hachage simple avec chaînage pour la gestion des collisions."""

    def __init__(self, capacity=16):
        self.capacity = capacity
        self.size = 0
        # Chaque bucket est une liste de paires (clé, valeur)
        self.buckets = [[] for _ in range(self.capacity)]

    def _index(self, key):
        """Calcule l'indice du bucket pour une clé donnée."""
        return hash(key) % self.capacity

    def set(self, key, value):
        """Insère ou met à jour une paire (clé, valeur). O(1) amorti."""
        idx = self._index(key)
        bucket = self.buckets[idx]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)  # mise à jour
                return
        bucket.append((key, value))  # insertion
        self.size += 1

    def get(self, key, default=None):
        """Récupère la valeur associée à une clé. O(1) amorti."""
        idx = self._index(key)
        for k, v in self.buckets[idx]:
            if k == key:
                return v
        return default

    def delete(self, key):
        """Supprime une paire (clé, valeur). O(1) amorti."""
        idx = self._index(key)
        bucket = self.buckets[idx]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self.size -= 1
                return True
        return False

    def load_factor(self):
        """Retourne le facteur de charge : nb éléments / capacité."""
        return self.size / self.capacity

    def collision_stats(self):
        """Statistiques sur les collisions."""
        lengths = [len(b) for b in self.buckets]
        collisions = sum(max(0, l - 1) for l in lengths)
        max_chain = max(lengths)
        return {"collisions": collisions, "max_chain": max_chain}


In [ ]:
# --- Test ---

# VOTRE CODE

ht = HashTable(

In [ ]:
# Visualiser la répartition dans les buckets
import matplotlib.pyplot as plt

# VOTRE CODE
# Remplir avec plus de données pour voir les collisions



#### Adressage ouvert : le cas du *linear probing*

C’est l’autre méthode de référence. Par exemple c’est celle qui est utilisée pour les dictionnaires Python qui utilisent une variante optimisée de l’adressage ouvert plutôt que le chaînage.

On appelle adressage ouvert les méthodes où toutes les valeurs sont stockées **directement dans le tableau**, et en cas de collision **on cherche une autre case** selon une certaine stratégie de sondage. Voici différentes stratégies d’adressage ouvert : *Linear probing, Quadratic probing, Double hashing, Robin Hood hashing, Hopscotch hashing…*

La stratégie la plus intuitive (simple) que nous allons présenter ici est le *linear probing*, où, en cas de collision sur la case i, on essaie i+1, puis i+2, etc. jusqu'à trouver une case libre.

Insertion de "alice" → hash/compression = 2 → case 2 libre → on place ici
Insertion de "charlie" → hash/compression = 2 → case 2 occupée → on essaie 3 → libre → on place ici

| Indice | Valeur |
|--------|--------|
| [0] | → vide |
| [1] | → vide |
| [2] | → ("alice", 42) |
| [3] | → ("charlie", 7)   ← déplacé par linear probing |
| [4] | → vide |

Cela a un impact sur les opération classiques :

Par exemple, recherche de "charlie" : hash = 2, on trouve "alice" ≠ "charlie", on avance en 3, on trouve "charlie" ! L’accès n’est plus immédiat, il y a une étape en plus. L’accès nécessite parfois plusieurs sondages successifs avant de trouver la bonne case, mais on considère toujours le coût moyen comme O(1).

Alors que la supression est simple avec le chaînage, elle devient assez délicate avec l’adressage ouvert, car supprimer une case peut casser la chaîne de sondage. C’est pour cela que l’on va utiliser des marqueur `DELETED` ou `tombstone` (cf. sujet de mémoire n°1).

L'avantage de l’adressage ouvert sur le chaînage est la **localité mémoire** : toutes les données sont dans le même tableau contigu, ce qui permet de mieux exploiter les lignes de cache CPU, ce qui réduit les accès mémoire coûteux. L'inconvénient est le **clustering** : les collisions tendent à créer des plages de cases occupées consécutives, ce qui allonge les séquences de sondage. C’est particulièrement vrai en linear probing qui, par design, fait grossir rapidement les groupes d’éléments contigus : c’est ce que l’on appelle le clustering primaire.

> CPython implémente les `dict` et `set` avec de **l’adressage ouvert** (et non du chaînage), précisément pour bénéficier de la localité mémoire (mais pas avec du linear probing).

#### Le facteur de charge

Les deux stratégies se dégradent quand le tableau est trop rempli. En pratique néanmoins, la stratégie de chaînage reste efficace même si $\alpha>1$, mais cette situation est beaucoup plus défavorable rapidement pour l’adressage ouvert. En effet, lors que le tableau est presque totalement remplit, le temps de sondage est fortement augmenté pour trouver une case vide. Python maintient donc le facteur de charge $\alpha$ en dessous de ~2/3 en redimensionnant automatiquement le tableau interne quand ce seuil est dépassé. C'est le même mécanisme d'allocation que pour les tableaux dynamiques. C'est ce redimensionnement qui garantit que les collisions restent rares et que le O(1) moyen est préservé.

### Exercice : Implémenter une HashTable avec linear probing (version simple)

Réimplémentez la classe `HashTable` en utilisant le **linear probing**, sans gestion des suppressions dans un premier temps.

**Contraintes :**
- `self.buckets` est un tableau plat de taille `capacity`, initialisé à `None`
- `_probe(key)` parcourt la séquence de sondage (en avançant circulairement dans le tableau ((idx + 1) % capacity) et retourne :
  - l'index de la case contenant `key` si elle est trouvée
  - le premier index `None` rencontré si la clé est absente (En linear probing, lorsqu’on rencontre une case None, on sait que la clé recherchée n’existe pas dans la table.)
- Implémentez `set()`, `get()` et `load_factor()`
- Refusez toute insertion si elle fait dépasser le facteur de charge de 0.66
- On n'implémente pas `delete()` pour l'instant

In [ ]:
class HashTableLP:
    """Table de hachage avec linear probing – version sans suppression."""

    def __init__(self, capacity=16):
        self.capacity = capacity
        self.size = 0
        self.buckets = [None] * self.capacity

    def _index(self, key):
        return hash(key) % self.capacity

    def _probe(self, key):
        """
        Retourne un tuple (found, index) où :
        - found=True  → key trouvée à index
        - found=False → key absente, index est le premier None disponible
        """
        # À compléter
        pass

    def set(self, key, value):
        """Insère ou met à jour une paire (clé, valeur)."""
        if self.load_factor() >= 0.66:
            raise Exception("Table pleine – facteur de charge dépassé")
        # À compléter
        pass

    def get(self, key, default=None):
        """Récupère la valeur associée à une clé."""
        # À compléter
        pass

    def load_factor(self):
        return self.size / self.capacity

**Bonus** : Ajouter une méthode collision_stats() comptant le nombre de clés qui ne sont pas à leur position naturelle.

### Exercice suite : Ajouter la suppression avec tombstones

La version précédente ne supporte pas la suppression. Essayons naïvement de supprimer une clé en remplaçant sa case par `None` et observons ce qui se passe :

In [ ]:
ht = HashTableLP(capacity=4)

ht.set(2, "alice")
ht.set(6, "bob")   # collision : 2 % 4 == 6 % 4 == 2

ht.buckets[ht._probe(2)[1]] = None

print(ht.get(6))   # pourquoi retourne-t-il None ?

**Question** : pourquoi `get(6)` retourne-t-il `None` alors que la clé `6` est toujours dans la table (c’est la 2 qu’on a effacé) ?

Le probing s’arrête lorsqu’il rencontre `None`, car cela signifie :

> « aucune clé n’a jamais été insérée plus loin dans cette séquence ».

Modifiez la classe pour introduire une sentinelle `DELETED` (*tombstone*) et adaptez `_probe()` pour qu'elle :

* continue après un tombstone (ne pas s'arrêter),
* mémorise le premier tombstone rencontré,
* retourne :

  * la clé si elle est trouvée,
  * sinon le premier tombstone rencontré,
  * sinon le premier `None`.

En d’autres termes, `_probe()` doit distinguer :

* une case jamais utilisée (`None`),
* une case supprimée (`DELETED`).


In [ ]:
# Votre code


In [ ]:
# TEST de votre code

ht = HashTableLP(capacity=4)

# Toutes ces clés collisionnent :
# 2 % 4 == 6 % 4 == 10 % 4 == 2
ht.set(2, "alice")
ht.set(6, "bob")

# Suppression : création d'un tombstone
ht.delete(2)

# Vérifie que la clé 6 reste accessible
assert ht.get(6) == "bob", "La clé 6 devrait rester accessible après suppression"

# Mise à jour d'une clé déjà présente après un tombstone
# _probe() doit continuer après DELETED et retrouver la vraie clé
ht.set(6, "BOB_UPDATED")

# Vérifie qu'on a bien mis à jour la clé existante
assert ht.get(6) == "BOB_UPDATED", "La valeur de la clé 6 devrait être mise à jour"

# Vérifie qu'on n'a pas créé un doublon de clé
count_key_6 = sum(
    1
    for bucket in ht.buckets
    if bucket not in (None, ht.DELETED) and bucket[0] == 6
)

assert count_key_6 == 1, "Il ne doit pas y avoir de doublon pour la clé 6"

print("Test tombstone + mise à jour OK ✓")

#### Question pour approfondir

> Que se passe-t-il dans le cas où on insére un nouvel élément et qu’il tombe sur un `DELETED` : on remplace purement et simplement, ou est-ce que c’est plus compliqué que ça ?


#### Votre réponse :

-


### Remarque : Saturation en tombstones

Dans cette implémentation, le facteur de charge (`load_factor`) ne compte que les éléments réellement présents dans la table (`size`) et ignore les tombstones (`DELETED`). Cela simplifie le code, mais introduit une subtilité importante.

En effet, après de nombreuses suppressions, la table peut contenir très peu d’éléments actifs tout en étant remplie de tombstones. Or les tombstones restent présents dans les séquences de sondage : les recherches et insertions doivent continuer à les traverser. Les performances peuvent alors se dégrader fortement même si le facteur de charge semble faible.

Exemple :

* capacité = 100
* 5 clés actives
* 90 tombstones

Le facteur de charge vaut alors :

$$
\alpha = \frac{5}{100} = 0.05
$$

ce qui paraît excellent. Pourtant, les séquences de probing peuvent devenir très longues car la majorité des cases ont déjà été utilisées.

Les implémentations industrielles de tables de hachage surveillent donc souvent :

* le nombre d’éléments actifs,
* mais aussi le nombre de tombstones.

Lorsque trop de tombstones s’accumulent, la table est reconstruite (*rehashing*) afin de nettoyer les cases supprimées et restaurer de bonnes performances.


### Pour aller plus loin

[Un cours plus complet](https://helios2.mi.parisdescartes.fr/~lomn/Cours/AV/AlgoAvancee5_hashTables.pdf) qui aborde également le *quadratic probing*